In [1]:
data_dir = "/Users/user-macmini/Downloads/Multivariate_ts"

In [2]:
# %pip install aeon seaborn torch numpy pandas

In [3]:
import sys
import random
import numpy as np
import time
import gc
import warnings
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [4]:
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier as DTWClassifier
from aeon.classification.convolution_based import RocketClassifier, MultiRocketHydraClassifier, HydraClassifier
from aeon.classification.hybrid import HIVECOTEV2
from aeon.datasets import load_classification

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [5]:
def timeit(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"\tFunction {func.__name__} took {end_time - start_time:.4f} seconds to execute.")
        return result
    return wrapper

In [6]:
@timeit
def train_model(classifier, X_train, y_train):
    classifier.fit(X_train, y_train)
    return classifier

@timeit
def eval_model(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)
    return accuracy_score(y_test, y_pred)


In [ ]:
UEA_MTSC30 = ['EthanolConcentration',
              'FaceDetection',
              'Handwriting',
              'Heartbeat',
              'JapaneseVowels',
              'PEMS-SF',
              'SelfRegulationSCP1',
              'SelfRegulationSCP2',
              'SpokenArabicDigits',
              'UWaveGestureLibrary',
              'ArticularyWordRecognition',
              'AtrialFibrillation',
              'BasicMotions',
              'CharacterTrajectories',
              'Cricket',
              'DuckDuckGeese',
              'Epilepsy',
              'ERing',
              'FingerMovements',
              'HandMovementDirection',
              'InsectWingbeat',
            #   'Libras',
            #   'LSST',
            #   'MotorImagery',
            #   'NATOPS',
            #   'PenDigits',
            #   'PhonemeSpectra',
            #   'RacketSports',
            #   'StandWalkJump',
            #   'EigenWorms',
              ]

SEED = 2

models = {
    # 'DTW': DTWClassifier(n_jobs=8),  # result will be same regardless of random_state
    'Rocket': RocketClassifier(random_state=SEED, n_jobs=8),
    'Hydra': HydraClassifier(random_state=SEED, n_jobs=8),
    'Hive-Cote v2': HIVECOTEV2(time_limit_in_minutes=0.2, random_state=SEED, n_jobs=8),
    'MultiRocket-Hydra': MultiRocketHydraClassifier(random_state=SEED, n_jobs=8),
}
for dname in UEA_MTSC30:
    print(f"===== {dname} =====")
    X_train, y_train = load_classification(dname, split="train", extract_path=data_dir, load_equal_length=True)
    X_test, y_test = load_classification(dname, split="test", extract_path=data_dir, load_equal_length=True)
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    for model in models:
        print(f"\t===== {model} =====")
        # paddding to make the length >= 9
        if X_train.shape[2] < 10 and model == 'MultiRocket-Hydra':
            X_train = np.pad(X_train, ((0, 0), (0, 0), (0, 10 - X_train.shape[2])), 'edge')
            X_test = np.pad(X_test, ((0, 0), (0, 0), (0, 10 - X_test.shape[2])), 'edge')
            print(f"\tPadded Train shape: {X_train.shape}, Padded Test shape: {X_test.shape}")
        
        random.seed(SEED)
        np.random.seed(SEED)
        models[model] = train_model(models[model], X_train, y_train)
        accuracy = eval_model(models[model], X_test, y_test)
        print(f"\t{model} on {dname}: {accuracy * 100} %")
        print(f"\t{model} on {dname} object size is {sys.getsizeof(models[model])/1024} KB")
        models[model].reset(keep=None)
        gc.collect()

===== EthanolConcentration =====
Train shape: (261, 3, 1751), Test shape: (263, 3, 1751)
	===== Rocket =====
	Function train_model took 10.1553 seconds to execute.
	Function eval_model took 11.3127 seconds to execute.
	Rocket on EthanolConcentration: 41.06463878326996 %
	Rocket on EthanolConcentration object size is 0.046875 KB
	===== Hydra =====
	Function train_model took 12.6727 seconds to execute.
	Function eval_model took 12.0329 seconds to execute.
	Hydra on EthanolConcentration: 52.47148288973384 %
	Hydra on EthanolConcentration object size is 0.046875 KB
	===== Hive-Cote v2 =====
	Function train_model took 68.1670 seconds to execute.
	Function eval_model took 62.2252 seconds to execute.
	Hive-Cote v2 on EthanolConcentration: 47.90874524714829 %
	Hive-Cote v2 on EthanolConcentration object size is 0.046875 KB
	===== MultiRocket-Hydra =====
	Function train_model took 14.4834 seconds to execute.
	Function eval_model took 14.0082 seconds to execute.
	MultiRocket-Hydra on EthanolConc

In [ ]:
UEA_MTSC30 = [
              'Libras',
              'LSST',
              'MotorImagery',
              'NATOPS',
              'PenDigits',
              'PhonemeSpectra',
              'RacketSports',
              'StandWalkJump',
              'EigenWorms',
            ]

SEED = 2
models = {
    # 'DTW': DTWClassifier(n_jobs=8),  # result will be same regardless of random_state
    'Rocket': RocketClassifier(random_state=SEED, n_jobs=8),
    'Hydra': HydraClassifier(random_state=SEED, n_jobs=8),
    'Hive-Cote v2': HIVECOTEV2(time_limit_in_minutes=0.2, random_state=SEED, n_jobs=8),
    'MultiRocket-Hydra': MultiRocketHydraClassifier(random_state=SEED, n_jobs=8),
}
for dname in UEA_MTSC30:
    print(f"===== {dname} =====")
    X_train, y_train = load_classification(dname, split="train", extract_path=data_dir, load_equal_length=True)
    X_test, y_test = load_classification(dname, split="test", extract_path=data_dir, load_equal_length=True)
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    for model in models:
        print(f"\t===== {model} =====")
        # paddding to make the length >= 9
        if X_train.shape[2] < 10 and model == 'MultiRocket-Hydra':
            X_train = np.pad(X_train, ((0, 0), (0, 0), (0, 10 - X_train.shape[2])), 'edge')
            X_test = np.pad(X_test, ((0, 0), (0, 0), (0, 10 - X_test.shape[2])), 'edge')
            print(f"\tPadded Train shape: {X_train.shape}, Padded Test shape: {X_test.shape}")
        
        random.seed(SEED)
        np.random.seed(SEED)
        models[model] = train_model(models[model], X_train, y_train)
        accuracy = eval_model(models[model], X_test, y_test)
        print(f"\t{model} on {dname}: {accuracy * 100} %")
        print(f"\t{model} on {dname} object size is {sys.getsizeof(models[model])/1024} KB")
        models[model].reset(keep=None)
        gc.collect()

===== Libras =====
Train shape: (180, 2, 45), Test shape: (180, 2, 45)
	===== Rocket =====
	Function train_model took 0.4851 seconds to execute.
	Function eval_model took 0.3219 seconds to execute.
	Rocket on Libras: 90.0 %
	Rocket on Libras object size is 0.046875 KB
	===== Hydra =====
	Function train_model took 0.5514 seconds to execute.
	Function eval_model took 0.4564 seconds to execute.
	Hydra on Libras: 94.44444444444444 %
	Hydra on Libras object size is 0.046875 KB
	===== Hive-Cote v2 =====
	Function train_model took 12.2736 seconds to execute.
	Function eval_model took 7.9911 seconds to execute.
	Hive-Cote v2 on Libras: 91.66666666666666 %
	Hive-Cote v2 on Libras object size is 0.046875 KB
	===== MultiRocket-Hydra =====
	Function train_model took 0.9908 seconds to execute.
	Function eval_model took 0.6204 seconds to execute.
	MultiRocket-Hydra on Libras: 93.33333333333333 %
	MultiRocket-Hydra on Libras object size is 0.046875 KB
===== LSST =====
Train shape: (2459, 6, 36), Test